In [ ]:
import os
import getpass
from typing import TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

In [ ]:
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key: ")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.6, max_tokens=1500)

In [ ]:
class AgentState(TypedDict):
    student_name: str
    student_goals: str
    performance_data: dict
    learning_diagnosis: str
    study_plan: str
    resources: str
    practice_quiz: str

In [ ]:
def diagnose_student(state: AgentState):
    prompt = f"""You are a learning analytics expert.
Student: {state['student_name']}
Subject: {state['performance_data']['subject']}
Recent Scores: {state['performance_data']['recent_scores']}
Weak Topics: {state['performance_data']['weak_topics']}
Weekly Study Hours: {state['performance_data']['study_hours_per_week']}
Risk Level: {state['performance_data']['ml_risk_classification']}
Goal: {state['student_goals']}

Write a concise learning diagnosis identifying strengths, weaknesses, and risk verdict."""
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"learning_diagnosis": response.content}


def make_plan(state: AgentState):
    prompt = f"""You are an academic coach.
Diagnosis: {state['learning_diagnosis']}
Goal: {state['student_goals']}
Study hours/week: {state['performance_data']['study_hours_per_week']}

Create a 4-week study plan with specific daily tasks per week."""
    response = llm.invoke([SystemMessage(content="You are an expert study planner."), HumanMessage(content=prompt)])
    return {"study_plan": response.content}


def fetch_resources(state: AgentState):
    prompt = f"""Recommend 5 free learning resources for: {state['performance_data']['weak_topics']}
Subject: {state['performance_data']['subject']}
For each: name, real URL, one sentence on what it covers."""
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"resources": response.content}


def make_quiz(state: AgentState):
    prompt = f"""Write a 5-question multiple choice quiz testing: {state['performance_data']['weak_topics']}
Subject: {state['performance_data']['subject']}
For each question: question text, 4 options (A-D), correct answer with brief explanation."""
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"practice_quiz": response.content}

In [ ]:
workflow = StateGraph(AgentState)
workflow.add_node("Diagnose", diagnose_student)
workflow.add_node("Plan", make_plan)
workflow.add_node("Resources", fetch_resources)
workflow.add_node("Quiz", make_quiz)
workflow.set_entry_point("Diagnose")
workflow.add_edge("Diagnose", "Plan")
workflow.add_edge("Plan", "Resources")
workflow.add_edge("Resources", "Quiz")
workflow.add_edge("Quiz", END)
agent = workflow.compile()

In [ ]:
sample = {
    "student_name": "Arjun",
    "student_goals": "Score 75% in final Math exam.",
    "performance_data": {
        "subject": "Mathematics",
        "recent_scores": [40, 55, 48],
        "weak_topics": ["Algebra", "Integration"],
        "study_hours_per_week": 5,
        "ml_risk_classification": "At Risk"
    }
}

result = agent.invoke(sample)

print("DIAGNOSIS
", result["learning_diagnosis"])
print("
STUDY PLAN
", result["study_plan"])
print("
RESOURCES
", result["resources"])
print("
QUIZ
", result["practice_quiz"])